<span style="color: #6a737d; font-family: monospace;">
Created on Tue Feb 18 2025 17:46:38<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2025 Mukai (Tom Notch) Yu<br>
</span>

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

from itertools import islice

import torch
from torch.utils.data import DataLoader

from usf.dataset.pandora import PandoraDataset
from usf.network.model.object_detection import ObjectDetectionLightningModel
from usf.utils.files import read_file
from usf.utils.spherical_image import BatchSphericalImage
from usf.visualization.spherical_layer import visualize_spherical_channels
from usf.visualization.spherical_projection import visualize_spherical_image

## Read Config

In [ ]:
config = read_file("config/task/object_detection.yaml")

In [ ]:
pandora_dataset_base_path = "data/PANDORA"
augmentation = {
    "chroma_jitter": 0.5,
    "luma_jitter": 0.5,
    "gaussian_blur": 0.5,
    "gray_scale": 0.2,
    "horizontal_reflection": 0.5,
    "vertical_reflection": 0.5,
    "erase": 0.5,
    "rotation": 0.5,
}
meta = {
    "input": {
        "mean": [0.52076054, 0.46685297, 0.41337782],  # in [0, 1] scale
        "std": [0.23466274, 0.23835337, 0.25246194],  # in [0, 1] scale
        "range": [0.0, 255.0],
    }
}
seed = "Object Detection"
downsample_image_size = (960, 480)
output_vector = read_file("config/lens_normal_map/180_180_560_560.npy")
output_vector_mask = read_file("config/lens_normal_map/180_180_560_560_mask.npy")
batch_size = 4

In [ ]:
device = "cuda"
dtype = torch.float32
factory_kwargs = {"device": device, "dtype": dtype}

# PANDORA Dataset

In [ ]:
pandora_train_dataset = PandoraDataset(
    dataset_base_path=pandora_dataset_base_path,
    dataset_type="train",
    augmentation=augmentation,
    downsample_image_size=downsample_image_size,
    output_vector=output_vector,
    output_vector_mask=output_vector_mask,
    meta=meta,
    seed=seed,
)
pandora_train_dataloader = DataLoader(
    pandora_train_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=pandora_train_dataset.collate_fn,
)

In [ ]:
# Retrieve one batch from the DataLoader
batch_idx = 0
batch = pandora_train_dataset.move_batch_to(
    next(islice(pandora_train_dataloader, batch_idx, None)),
    device=device,
    dtype=dtype,
)

In [ ]:
visualize_spherical_image(batch["inputs"]["spherical_images"], point_size=5, fps=1)

In [ ]:
figs, spherical_images_vis = pandora_train_dataset.visualize_batch(batch)

In [ ]:
visualize_spherical_image(spherical_images_vis[0], point_size=5, fps=1)

## Load Trained Model

In [ ]:
model = ObjectDetectionLightningModel(config["model"]).to(device)
model.load_state_dict(
    torch.load("../data/object_detection/spherical-199-nrr-npl.ckpt")["state_dict"]
)

In [ ]:
batch["predicts"] = {}
batch["predicts"]["maps"] = model(batch["inputs"])

Save

In [ ]:
batch["predicts"]["maps"]["heatmaps"].save(
    "../data/object_detection/sample_heatmap.npz"
)
batch["predicts"]["maps"]["center_offsets"].save(
    "../data/object_detection/sample_center_offset.npz"
)
batch["predicts"]["maps"]["sizes"].save("../data/object_detection/sample_size.npz")
batch["predicts"]["maps"]["rotations"].save(
    "../data/object_detection/sample_rotation.npz"
)

Load

In [ ]:
batch["predicts"] = {}
batch["predicts"]["maps"] = {}
batch["predicts"]["maps"]["heatmaps"] = BatchSphericalImage(
    "../data/object_detection/sample_heatmap.npz"
).to(device=device, dtype=dtype)
batch["predicts"]["maps"]["center_offsets"] = BatchSphericalImage(
    "../data/object_detection/sample_center_offset.npz"
).to(device=device, dtype=dtype)
batch["predicts"]["maps"]["sizes"] = BatchSphericalImage(
    "../data/object_detection/sample_size.npz"
).to(device=device, dtype=dtype)
batch["predicts"]["maps"]["rotations"] = BatchSphericalImage(
    "../data/object_detection/sample_rotation.npz"
).to(device=device, dtype=dtype)

# Generate Ground Truth Supervision Maps

In [ ]:
batch["labels"]["maps"] = pandora_train_dataset.generate_gt_maps(
    batch["labels"]["converted_rbfovs"],
    batch["predicts"]["maps"]["heatmaps"].vector,
    n_points=3,
)

## Visualize Heatmap Category by Category

In [ ]:
visualize_spherical_image(
    visualize_spherical_channels(
        batch["labels"]["maps"]["heatmaps"],
        value_range=(0.0, 1.0),
    )[0],
    point_size=5,
    fps=1,
)

## Visualize Center Offset

In [ ]:
visualize_spherical_image(
    visualize_spherical_channels(
        batch["labels"]["maps"]["masks"],
        binary=True,
    )[0],
    point_size=5,
    fps=1,
)

## Extract Raw RBFoV

In [ ]:
batch["predicts"]["raw_rbfovs"] = pandora_train_dataset.extract_raw_rbfovs(
    batch["predicts"]["maps"], probability_threshold=0.1
)

In [ ]:
batch["labels"]["raw_rbfovs"] = pandora_train_dataset.extract_raw_rbfovs(
    batch["labels"]["maps"], probability_threshold=0.1
)

You can see there're still situations where big objects overrides small object's type, we can suppress them within the same category

## Non Maximum Suppression

In [ ]:
batch["predicts"]["suppressed_rbfovs"] = pandora_train_dataset.non_maximum_suppression(
    batch["predicts"]["raw_rbfovs"],
    max_rbfov_per_category=20,
    sigma=0.25,
    score_threshold=0.5,
)

In [ ]:
batch["labels"]["suppressed_rbfovs"] = pandora_train_dataset.non_maximum_suppression(
    batch["labels"]["raw_rbfovs"],
    max_rbfov_per_category=20,
    sigma=0.25,
    score_threshold=0.5,
)

## Benchmark

mean Average Precision

In [ ]:
def get_dominant_category(rbfovs: torch.Tensor):
    """
    Returns the dominant category (most frequent) from the RBFoVs.

    Args:
        rbfovs (torch.Tensor): tensor, one per image, with each tensor's shape
            (num_rbfovs, 7) in the format [θ, φ, α, β, γ, category, confidence].

    Returns:
        int: Dominant category or None if no ground truth exists.
    """
    if rbfovs.numel() == 0:
        return None
    # Convert the category column to integers.
    categories = rbfovs[:, 5].long()
    dominant_category = categories.mode()[0].item()
    return dominant_category

In [ ]:
model.benchmark(
    batch["predicts"]["suppressed_rbfovs"],
    batch["predicts"]["suppressed_rbfovs"],
    iou_threshold=0.5,
)

In [ ]:
model.benchmark(
    batch["labels"]["converted_rbfovs"],
    batch["labels"]["converted_rbfovs"],
    iou_threshold=0.5,
)

In [ ]:
model.benchmark(
    batch["predicts"]["suppressed_rbfovs"],
    batch["labels"]["converted_rbfovs"],
    iou_threshold=0.2,
)

## Visualize Results

In [ ]:
predict_figs, _ = pandora_train_dataset.visualize_batch(
    batch={
        "images": batch["images"],
        "spherical_images": batch["spherical_images"],
        "labels": {"converted_rbfovs": batch["predicts"]["suppressed_rbfovs"]},
    },
    num_vis=batch_size,
)

In [ ]:
gt_figs, _ = pandora_train_dataset.visualize_batch(
    batch={
        "images": batch["images"],
        "spherical_images": batch["spherical_images"],
        "labels": {"converted_rbfovs": batch["labels"]["suppressed_rbfovs"]},
    },
    num_vis=batch_size,
)